<a href="https://colab.research.google.com/github/bored-shinigami2805/CVPR-SLM-fine-tuning/blob/main/02_chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U "transformers>=4.45" "peft>=0.13" "accelerate>=0.34"
import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
import os, zipfile
ADAPTER_DIR = "qwen2.5-3b-papers-lora"
if not os.path.isdir(ADAPTER_DIR):
    if os.path.exists(ADAPTER_DIR + ".zip"):
        with zipfile.ZipFile(ADAPTER_DIR + ".zip") as z:
            z.extractall(ADAPTER_DIR)
    else:
        try:
            from google.colab import files
            print("Upload the adapter zip (qwen2.5-3b-papers-lora.zip) ...")
            up = files.upload()
            name = list(up)[0]
            with zipfile.ZipFile(name) as z:
                z.extractall(ADAPTER_DIR)
        except Exception as e:
            print("Provide the adapter folder/zip manually. |", e)
assert os.path.isdir(ADAPTER_DIR), "adapter folder not found"
print("Adapter ready at", ADAPTER_DIR)


In [ ]:
import peft.tuners.lora.torchao as _tao
_tao.is_torchao_available = lambda: False

## 3. Load base model + adapter

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_DIR = "qwen2.5-3b-papers-lora"

tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto"
)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
print("Model loaded.")


In [ ]:
SYSTEM = (
    "You are a research assistant that answers questions ONLY about five specific "
    "computer-vision research papers: (1) ShutterMuse: Capture-Time Photography Guidance "
    "with MLLMs; (2) Shift-Variant Image Degradation and Restoration Using Singular Value "
    "Decomposition; (3) Naturalness Predicts but Does Not Cause Transferability in Image "
    "Encodings of Real-World Streams; (4) In-context Region-based Drag (ICRDrag); and "
    "(5) How Robust is OCR-Reasoning? (OCR-Robust). Answer strictly from the content of "
    "these papers. If a question is not covered by these papers, reply exactly: "
    "\"I don't have information.\""
)


In [ ]:
@torch.no_grad()
def generate(history, max_new_tokens=256):
    msgs = [{"role": "system", "content": SYSTEM}] + history
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         temperature=None, top_p=None, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, inputs.shape[1]:], skip_special_tokens=True).strip()


In [ ]:
@torch.no_grad()
def generate(history, max_new_tokens=256):
    msgs = [{"role": "system", "content": SYSTEM}] + history
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

## 6. Chat loop
Type your question and press Enter. Type `quit`, `exit`, or leave empty to stop. Type `reset` to clear the conversation.

In [ ]:
\history = []
print("Paper SLM ready. Ask about the 5 papers. (quit / exit to stop, reset to clear)\n")
while True:
    try:
        q = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n[ended]"); break
    if q.lower() in ("quit", "exit", ""):
        print("[ended]"); break
    if q.lower() == "reset":
        history = []; print("[history cleared]\n"); continue
    history.append({"role": "user", "content": q})
    ans = generate(history)
    history.append({"role": "assistant", "content": ans})
    print("Bot:", ans, "\n")


## 7.  Batch ask without the loop

In [ ]:
def ask(q):
    return generate([{"role": "user", "content": q}])

for q in ["Summarize the OCR-Robust paper.",
          "What rewards does ShutterMuse use in GRPO?",
          "What is the WorldStream corpus?",
          "What's a good recipe for dinner?"]:
    print("Q:", q); print("A:", ask(q)); print("-"*60)
